# 03 — Il dataset finale

Dagli articoli puliti (`01`) e dalle reti (`02`) al dataset su cui si classifica. Le scelte, fissate
**prima** di addestrare i modelli e uguali per tutti:

1. **Gruppi** = reti trovate da `02` **∪** testate con lo stesso owner NewsGuard. Le reti di `02` trovano
   parentele che l'owner non riporta (proprietà cambiate, gestioni comuni, contenuti condivisi); l'owner copre
   le reti che non lasciano tracce nell'HTML (per esempio i giornali Boone, legati solo dal fornitore del CMS).
   Un gruppo sta tutto nel training o tutto nel test.
2. **Copie tra gruppi diversi**: di un testo pubblicato da testate di gruppi diversi si tiene la copia più
   vecchia (la presumibile originale) e si tolgono le altre. Così lo stesso testo non sta mai nel training e nel
   test; le copie dentro un gruppo restano.
3. **(Versione 2) Due insiemi di testate.**
   - **Valutate** (il test): almeno `MIN_ARTICOLI` articoli e al massimo `MAX_TESTATE_PER_GRUPPO` testate per gruppo,
     scelte a caso (seed fisso, senza guardare le etichette). Sono le stesse regole della prima versione, così i
     numeri restano confrontabili: senza tetto un solo gruppo (Natural News) farebbe 239 delle 319 testate $l_1$.
   - **Di solo training**: tutte le altre testate dei gruppi, con almeno `MIN_ARTICOLI_TRAINING` articoli. Non
     vanno mai nel test, ma insegnano ai modelli: da 304 a circa 860 testate in addestramento.
4. **Peso** di ogni testata nell'addestramento: $\min(1,\ $`MAX_TESTATE_PER_GRUPPO`$/n)$, con $n$ le testate del suo
   gruppo. Un gruppo pesa al più quanto 5 testate: Natural News non domina $l_1$ anche se il modello vede tutte le
   sue testate. I transformer, invece del peso, pescano a ogni epoca al più 5 testate per gruppo (sempre diverse).

Qui, per la prima volta, si usano le etichette: per descrivere il dataset e per stratificare i fold.

- **fold per gruppo** (la valutazione principale), stratificati per livello;
- **fold per testata** (solo per confronto): i gruppi possono essere spezzati.

**Sul server** (serve l'HTML grezzo, che sta solo lì):

```bash
cd ~/John/WebScience2027/Versione2/1-data-cleaning
screen -dmS cleaning papermill -k johnvenv 03_dataset_finale.ipynb ../Results/1-data-cleaning/esecuzioni/03_dataset_finale.ipynb
```

In [ ]:
import os

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

from comune import DERIVED, LIVELLI, NEWSGUARD, RESULTS, SEED, VERSIONI, normalizza_owner

os.makedirs(VERSIONI, exist_ok=True)

In [ ]:
MIN_ARTICOLI = 20              # un voto di maggioranza su pochi articoli non è affidabile
MAX_TESTATE_PER_GRUPPO = 5     # nessun gruppo pesa più di qualche testata (nel test; nel training con i pesi)
MIN_ARTICOLI_TRAINING = 5      # (Versione 2) le testate con meno articoli servono comunque per l'addestramento

In [ ]:
art = pd.read_csv(DERIVED + "01_articoli_puliti.csv", parse_dates=["data"])
reti = pd.read_csv(RESULTS + "02_reti.csv").set_index("domain")
ng = pd.read_csv(NEWSGUARD, low_memory=False)
ng = ng.assign(d=ng.Domain.str.lower()).drop_duplicates("d").set_index("d")
passi = []


def passo(nome):
    passi.append({"passo": nome, "articoli": len(art), "testate": art.domain.nunique()})
    print(f"{nome}: {len(art)} articoli, {art.domain.nunique()} testate")


passo("articoli puliti (01)")

## 1. I gruppi: reti ∪ owner

In [ ]:
owner = pd.Series({d: normalizza_owner(ng.Owner.get(d, "")) for d in reti.index})
G = nx.Graph(); G.add_nodes_from(reti.index)
for _, g in reti.groupby("rete"):
    G.add_edges_from(zip(g.index, g.index[1:]))
noti = owner[owner != ""]
for _, g in noti.groupby(noti):
    G.add_edges_from(zip(g.index, g.index[1:]))
gruppo = {d: i for i, c in enumerate(sorted(nx.connected_components(G), key=len, reverse=True)) for d in c}
print(f"reti di 02: {reti.rete.nunique()} -> gruppi con l'owner: {len(set(gruppo.values()))}")
uniti = pd.DataFrame({"rete": reti.rete, "gruppo": pd.Series(gruppo)})
print("gruppi che l'owner ha unito a partire da più reti:")
for g_id, g in uniti.groupby("gruppo"):
    if g.rete.nunique() > 1:
        print(f"  gruppo {g_id}: {len(g)} testate da {g.rete.nunique()} reti, owner {owner[g.index].replace('', '?').value_counts().head(3).to_dict()}")
art["gruppo"] = art.domain.map(gruppo)

## 2. Copie tra gruppi diversi

In [ ]:
copie = pd.read_csv(DERIVED + "01_copie_tra_testate.csv")
presenti = set(art.article_id)
copie = copie[copie.articolo_a.isin(presenti) & copie.articolo_b.isin(presenti)]
C = nx.Graph(); C.add_edges_from(zip(copie.articolo_a, copie.articolo_b))
info = art.set_index("article_id")
via, famiglie_tra_gruppi = set(), 0
for fam in nx.connected_components(C):
    fam = sorted(fam, key=lambda x: (info.data[x], x))           # la più vecchia per prima
    if len({info.gruppo[x] for x in fam}) > 1:
        famiglie_tra_gruppi += 1
        primo = info.gruppo[fam[0]]
        via |= {x for x in fam if info.gruppo[x] != primo}
print(f"famiglie di copie: {nx.number_connected_components(C)}, di cui tra gruppi diversi {famiglie_tra_gruppi}; articoli tolti: {len(via)}")
art = art[~art.article_id.isin(via)]
passo("2. senza copie tra gruppi diversi")

## 3. Testate con abbastanza articoli, e il tetto per gruppo

In [ ]:
n = art.groupby("domain").size()
art = art[art.domain.map(n) >= MIN_ARTICOLI_TRAINING]
passo(f"3. testate con almeno {MIN_ARTICOLI_TRAINING} articoli (insieme di training)")
testate = art.groupby("domain").agg(gruppo=("gruppo", "first"), articoli=("article_id", "size")).reset_index()
idonee = testate[testate.articoli >= MIN_ARTICOLI]
prima = idonee.groupby("gruppo").size()
valutate = set(idonee.sample(frac=1, random_state=SEED).groupby("gruppo").head(MAX_TESTATE_PER_GRUPPO).domain)
testate["valutazione"] = testate.domain.isin(valutate)
testate["testate_nel_gruppo"] = testate.groupby("gruppo").domain.transform("size")
testate["peso"] = np.minimum(1.0, MAX_TESTATE_PER_GRUPPO / testate.testate_nel_gruppo)
print("gruppi oltre il tetto nel test:", {int(g): int(k) for g, k in prima[prima > MAX_TESTATE_PER_GRUPPO].sort_values(ascending=False).items()})
art["valutazione"] = art.domain.isin(valutate)
passi.append({"passo": f"4. testate valutate: almeno {MIN_ARTICOLI} articoli e al massimo {MAX_TESTATE_PER_GRUPPO} per gruppo",
              "articoli": int(art.valutazione.sum()), "testate": len(valutate)})
print(f"testate valutate: {len(valutate)} ({art.valutazione.sum()} articoli); di solo training: {(~testate.valutazione).sum()} "
      f"({(~art.valutazione).sum()} articoli); peso medio delle testate di training {testate.peso.mean():.2f}")

## Etichette e composizione

In [ ]:
art["score"] = art.domain.map(ng.Score)
art["label"] = pd.cut(art.score, [-1, 39, 59, 74, 99, 100], labels=[0, 1, 2, 3, 4]).astype(int)
art["rete"] = art.domain.map(reti.rete)
testate = testate.merge(art.groupby("domain").agg(label=("label", "first"), rete=("rete", "first")).reset_index(), on="domain")
testate["testate_valutate_nel_gruppo"] = testate.groupby("gruppo").valutazione.transform("sum")
L = testate.label.map(dict(enumerate(LIVELLI)))
V = testate.valutazione


def composizione(sel, art_sel):
    c = pd.DataFrame({
        "testate": L[sel].value_counts(),
        "articoli": art_sel.label.map(dict(enumerate(LIVELLI))).value_counts(),
        "in un gruppo con altre": L[sel & (testate.testate_nel_gruppo > 1)].value_counts(),
        "gruppi distinti": testate[sel].groupby(L[sel]).gruppo.nunique(),
    }).reindex(LIVELLI).fillna(0).astype(int)
    c.loc["totale"] = [int(sel.sum()), len(art_sel), int((sel & (testate.testate_nel_gruppo > 1)).sum()), testate[sel].gruppo.nunique()]
    return c


comp = composizione(V, art[art.valutazione])
comp.to_csv(RESULTS + "03_composizione.csv")
comp_tr = composizione(V | ~V, art)
comp_tr["peso totale"] = list(testate.groupby(L).peso.sum().reindex(LIVELLI).round(1)) + [round(testate.peso.sum(), 1)]
comp_tr.to_csv(RESULTS + "03_composizione_training.csv")
pd.DataFrame(passi).set_index("passo").to_csv(RESULTS + "03_passi.csv")
display(pd.DataFrame(passi).set_index("passo"))
print("testate valutate (il test):"); display(comp)
print("tutte le testate (il training):"); display(comp_tr)

## I fold

In [ ]:
# i fold si fanno sulle testate valutate, come nella prima versione; le testate di solo training seguono il loro gruppo
testate = testate.sort_values("domain").reset_index(drop=True)
val = testate[testate.valutazione].reset_index(drop=True)
sg = StratifiedGroupKFold(5, shuffle=True, random_state=SEED); val["fold_gruppo"] = 0
for i, (_, idx) in enumerate(sg.split(val.index, val.label, groups=val.gruppo), 1):
    val.loc[idx, "fold_gruppo"] = i
sk = StratifiedKFold(5, shuffle=True, random_state=SEED); val["fold_testata"] = 0
for i, (_, idx) in enumerate(sk.split(val.index, val.label), 1):
    val.loc[idx, "fold_testata"] = i
fold_del_gruppo = val.groupby("gruppo").fold_gruppo.first()
rng = np.random.default_rng(SEED)
senza = sorted(set(testate.gruppo) - set(fold_del_gruppo.index))      # gruppi senza testate valutate: un fold a caso
fold_del_gruppo = pd.concat([fold_del_gruppo, pd.Series(rng.integers(1, 6, len(senza)), index=senza)])
testate["fold_gruppo"] = testate.gruppo.map(fold_del_gruppo).astype(int)
testate["fold_testata"] = testate.domain.map(val.set_index("domain").fold_testata)
testate.loc[testate.fold_testata.isna(), "fold_testata"] = rng.integers(1, 6, int(testate.fold_testata.isna().sum()))
testate["fold_testata"] = testate.fold_testata.astype(int)
assert testate.groupby("gruppo").fold_gruppo.nunique().max() == 1
print("testate valutate per fold e livello:")
display(pd.crosstab(val.fold_gruppo, val.label.map(dict(enumerate(LIVELLI))), margins=True, margins_name="tot"))
print("tutte le testate (training) per fold e livello:")
pd.crosstab(testate.fold_gruppo, testate.label.map(dict(enumerate(LIVELLI))), margins=True, margins_name="tot")

In [ ]:
art[["article_id", "domain", "gruppo", "rete", "label", "score", "valutazione", "data", "titolo", "autore", "url", "text"]].to_csv(
    VERSIONI + "dataset_finale.csv", index=False)
testate[["domain", "label", "gruppo", "rete", "testate_nel_gruppo", "testate_valutate_nel_gruppo", "articoli", "valutazione", "peso",
         "fold_gruppo", "fold_testata"]].to_csv(VERSIONI + "fold_finale.csv", index=False)
print("salvati dataset_finale.csv e fold_finale.csv in", VERSIONI)